# Unit Test Generation — Autoresearch
PhD Research: Plain LLM vs Simple RAG vs Iterative Critique RAG for unit test generation.

**Steps:**
1. Mount Google Drive
2. Install dependencies
3. Install & start Ollama
4. Clone repo
5. One-time setup (dataset + knowledge base)
6. Run a single experiment (quick test)
7. Run all 12 experiments × 3 models (full PhD comparison)
8. Generalizability analysis (do rankings hold across models?)
9. Visualize results
10. Cross-task comparison (RQ4)
11. Push results to GitHub

## Step 1: Mount Google Drive
**Run this first.** All results, logs, charts, and checkpoints are saved to both local Colab storage and Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# All PhD outputs go here — edit if you want a different Drive folder
DRIVE_DIR = '/content/drive/MyDrive/PhD_autoresearch'

LOGS_DIR        = os.path.join(DRIVE_DIR, 'logs')
PLOTS_DIR       = os.path.join(DRIVE_DIR, 'plots')
RESULTS_DIR     = os.path.join(DRIVE_DIR, 'results')
CHECKPOINTS_DIR = os.path.join(DRIVE_DIR, 'checkpoints')

for d in [LOGS_DIR, PLOTS_DIR, RESULTS_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)

def save_to_both(src, filename, subdir='results'):
    """Copy a file to both local repo dir and Drive."""
    if not os.path.exists(src):
        print(f'  SKIP {filename} — not found')
        return
    drive_path = os.path.join(DRIVE_DIR, subdir, filename)
    shutil.copy(src, drive_path)
    print(f'  {filename} → local + Drive')

print('Drive mounted.')
print(f'  Local:  /content/autoresearch/')
print(f'  Drive:  {DRIVE_DIR}/')

## Step 2: Install Python dependencies

In [ ]:
!pip install -q ollama datasets sentence-transformers scikit-learn nltk rouge beautifulsoup4 numpy pandas matplotlib scipy

## Step 3: Install Ollama and pull models
Pulls all 3 models needed for the multi-model experiment.
On A100 (40GB): all 3 fit comfortably.

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print('Ollama server started.')

In [ ]:
# Models for multi-model generalizability study
# Matches the models used in the Docstring RAG experiment for cross-task RQ4
# A100 (40GB): all 3 fit
# T4  (15GB):  use llama3.2:latest + phi3:mini only

MODELS = [
    'llama3.2:latest',   # 3B  — fast baseline
    'phi4:14b',          # 14B — mid-size, matches Docstring experiment
    'qwen2.5:14b',       # 14B — mid-size, matches Docstring experiment
]

for model in MODELS:
    print(f'\nPulling {model}...')
    !ollama pull {model}

print('\nAll models ready.')
!ollama list

## Step 4: Clone the repo

In [ ]:
REPO_DIR = '/content/autoresearch'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/balajivenky06/autoresearch.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!ls

## Step 5: One-time setup (download dataset + build knowledge base)
Only needs to run once per Colab session.

In [ ]:
!python prepare_unitest.py

## Step 6: Configure and run a single experiment (quick test)
Use this to verify everything works before running the full multi-model sweep.

In [ ]:
import re, ast

VALID_METHODS    = {'plain_llm', 'simple_rag', 'iterative_critique'}
VALID_REASONINGS = {'base', 'cot', 'tot', 'got'}

def set_experiment(method='plain_llm', reasoning='base', model=MODELS[0]):
    """Update train_unitest.py config in-place."""
    assert method in VALID_METHODS, f'method must be one of {VALID_METHODS}'
    assert reasoning in VALID_REASONINGS, f'reasoning must be one of {VALID_REASONINGS}'

    with open('train_unitest.py', 'r') as f:
        content = f.read()

    content = re.sub(r'^METHOD\s*=.*$',         'METHOD    = "' + method    + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^REASONING\s*=.*$',       'REASONING = "' + reasoning + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^GENERATOR_MODEL\s*=.*$', 'GENERATOR_MODEL = "' + model + '"', content, flags=re.MULTILINE)
    content = re.sub(r'^HELPER_MODEL\s*=.*$',    'HELPER_MODEL    = "' + model + '"', content, flags=re.MULTILINE)

    try:
        ast.parse(content)
    except SyntaxError as e:
        raise RuntimeError('set_experiment produced invalid Python: ' + str(e))

    with open('train_unitest.py', 'w') as f:
        f.write(content)
    print(f'Config set: method={method}, reasoning={reasoning}, model={model}')

# Quick test with first model
set_experiment(method='plain_llm', reasoning='base', model=MODELS[0])

In [ ]:
log_local = 'run_unitest.log'
!python train_unitest.py 2>&1 | tee {log_local}

shutil.copy(log_local, os.path.join(LOGS_DIR, 'single_run.log'))
save_to_both('results_unitest.tsv', 'results_unitest.tsv')

!grep -E '^val_score:|^method:|^model:|^avg_|^Results appended' {log_local}

## Step 7: Run all 12 experiments × 3 models
**Full PhD comparison: 36 runs total (~6–10 hours on A100)**

- 3 models × 3 methods × 4 reasoning = 36 experiments
- All results accumulate in `results_unitest.tsv` (model column separates them)
- Checkpoints saved to Drive after every sample — safe to interrupt and resume

**On runtime reset:** re-run Steps 1–5, then re-run this cell. It resumes from the last checkpoint automatically.

In [ ]:
import subprocess, pandas as pd

EXPERIMENTS = [
    ('plain_llm',          'base'),
    ('plain_llm',          'cot'),
    ('plain_llm',          'tot'),
    ('plain_llm',          'got'),
    ('simple_rag',         'base'),
    ('simple_rag',         'cot'),
    ('simple_rag',         'tot'),
    ('simple_rag',         'got'),
    ('iterative_critique', 'base'),
    ('iterative_critique', 'cot'),
    ('iterative_critique', 'tot'),
    ('iterative_critique', 'got'),
]

all_results = []
sep = '=' * 60
total_runs = len(MODELS) * len(EXPERIMENTS)
run_num = 0

for model in MODELS:
    print(f'\n{sep}')
    print(f'  MODEL: {model}')
    print(sep)

    for method, reasoning in EXPERIMENTS:
        run_num += 1
        print(f'\n  [{run_num}/{total_runs}] {method}/{reasoning} — {model}')

        set_experiment(method=method, reasoning=reasoning, model=model)

        log_local = f'{model.replace(":", "-")}_{method}_{reasoning}.log'
        try:
            ret = subprocess.run(
                ['python', 'train_unitest.py'],
                capture_output=True, text=True, timeout=900
            )
            output = ret.stdout + ret.stderr
            with open(log_local, 'w') as f:
                f.write(output)
            shutil.copy(log_local, os.path.join(LOGS_DIR, log_local))

        except subprocess.TimeoutExpired:
            print('    TIMEOUT — skipping')
            all_results.append({
                'model': model, 'method': f'{method}/{reasoning}',
                'val_score': 0.0, 'status': 'timeout'
            })
            continue

        # Parse key metrics from stdout for in-memory summary
        row = {'model': model, 'method': f'{method}/{reasoning}', 'status': 'ok'}
        for line in output.splitlines():
            for key in ['val_score', 'avg_noise_rate', 'avg_faithfulness',
                        'avg_retrieval_secs', 'avg_llm_secs', 'avg_syntax',
                        'avg_edge', 'avg_assert_density', 'avg_semantic_sim',
                        'avg_rouge']:
                if line.strip().startswith(key + ':'):
                    try:
                        row[key] = float(line.split(':')[1].strip())
                    except Exception:
                        pass
        if row.get('val_score', 0.0) == 0.0:
            row['status'] = 'crash'
        all_results.append(row)
        print(f'    val_score: {row.get("val_score", 0):.4f}  [{row["status"]}]')

        # results_unitest.tsv is written by train_unitest.py — copy to Drive
        save_to_both('results_unitest.tsv', 'results_unitest.tsv')

# Final summary table across all models
df = pd.DataFrame(all_results)
print(f'\n{sep}')
print('  FINAL RESULTS — ALL MODELS')
print(sep)
if not df.empty and 'val_score' in df.columns:
    pivot = df.pivot_table(
        index='method', columns='model', values='val_score', aggfunc='max'
    ).round(4)
    print(pivot.to_string())

# Save full summary to both local and Drive
df.to_csv('summary_all_experiments.csv', index=False)
save_to_both('summary_all_experiments.csv', 'summary_all_experiments.csv')
print('\nDone.')

## Step 8: Generalizability analysis
**Key PhD question: do method rankings hold consistently across all 3 models?**

Computes Spearman rank correlation between models.
- ρ ≥ 0.8 across all pairs → findings generalize (model-agnostic)
- ρ < 0.8 → rankings are model-dependent (important nuance for thesis)

Outputs 4 charts + a written report for the thesis appendix.

In [ ]:
from IPython.display import Image, display

!python analyze_generalizability.py

generalizability_charts = [
    'val_score_by_model.png',
    'faithfulness_by_model.png',
    'rank_stability.png',
    'rank_correlation.png',
]

for fname in generalizability_charts:
    src = os.path.join('plots_generalizability', fname)
    save_to_both(src, f'generalizability_{fname}', subdir='plots')
    if os.path.exists(src):
        display(Image(src))

# Save the written report to Drive
save_to_both(
    'plots_generalizability/generalizability_report.txt',
    'generalizability_report.txt'
)

## Step 9: Visualize results
Generates 10 KPI charts (7 single-model + 3 cross-model) — saved to both local and Drive.

In [ ]:
from IPython.display import Image, display

!python visualize_unitest.py

charts = [
    # Single-model charts
    'heatmap.png',
    'grouped_bar.png',
    'radar.png',
    'per_metric_bar.png',
    'noise_rate.png',
    'cost_breakdown.png',
    'faithfulness.png',
    # Cross-model charts
    'model_val_score.png',
    'model_faithfulness.png',
    'model_rank_stability.png',
]

for fname in charts:
    src = os.path.join('plots_unitest', fname)
    save_to_both(src, fname, subdir='plots')
    if os.path.exists(src):
        display(Image(src))

## Step 10: Cross-task comparison — RQ4
Requires `results_docstring.tsv` from the RAG-Docstring repo.
Upload it to `PhD_autoresearch/results/` in Drive, then run this cell.

In [ ]:
# Load results_docstring.tsv from Drive
docstring_tsv_drive = os.path.join(RESULTS_DIR, 'results_docstring.tsv')
if os.path.exists(docstring_tsv_drive):
    shutil.copy(docstring_tsv_drive, 'results_docstring.tsv')
    print('results_docstring.tsv loaded from Drive.')
else:
    print(f'Not found: {docstring_tsv_drive}')
    print('Upload results_docstring.tsv to Drive first, then re-run this cell.')

In [ ]:
from IPython.display import Image, display

!python compare_tasks.py

compare_charts = ['faithfulness_by_task.png', 'noise_vs_faithfulness.png', 'pareto.png']
for fname in compare_charts:
    src = os.path.join('plots_compare', fname)
    save_to_both(src, f'compare_{fname}', subdir='plots')
    if os.path.exists(src):
        display(Image(src))

save_to_both('plots_compare/summary_table.tsv', 'cross_task_summary.tsv')

## Step 11: Push results to GitHub
Commits all results, logs, and charts to GitHub from Colab.

Get a token at: GitHub → Settings → Developer Settings → Personal Access Tokens → New token (scope: `repo`)

In [ ]:
# ---- Set your GitHub credentials ----
GITHUB_TOKEN = 'ghp_xxxxxxxxxxxxxxxxxxxx'   # paste your token here
GITHUB_EMAIL = 'your@email.com'
GITHUB_NAME  = 'Your Name'
# -------------------------------------

import subprocess

!git config user.email "{GITHUB_EMAIL}"
!git config user.name  "{GITHUB_NAME}"

# Unblock results_unitest.tsv from .gitignore
with open('.gitignore', 'r') as f:
    gi = f.read()
if 'results_unitest.tsv' in gi and '# results_unitest.tsv' not in gi:
    gi = gi.replace('results_unitest.tsv', '# results_unitest.tsv  (committed for PhD)')
    with open('.gitignore', 'w') as f:
        f.write(gi)
    print('Unblocked results_unitest.tsv from .gitignore')

# Stage all outputs
for f in ['results_unitest.tsv', 'summary_all_experiments.csv',
          'plots_unitest/', 'plots_compare/', 'plots_generalizability/']:
    !git add -f {f} 2>/dev/null || true

models_str = ', '.join(MODELS)
commit_msg = f'add experiment results: {models_str} — 36 runs (3 models × 12 methods)'
result = subprocess.run(['git', 'commit', '-m', commit_msg], capture_output=True, text=True)
print(result.stdout or result.stderr)

remote_url = f'https://{GITHUB_TOKEN}@github.com/balajivenky06/autoresearch.git'
push = subprocess.run(['git', 'push', remote_url, 'master'], capture_output=True, text=True)
if push.returncode == 0:
    print('Pushed to GitHub successfully.')
else:
    print('Push failed:', push.stderr)